# Day 2: Decorators - Advanced

## Decorators with Arguments

Decorators with arguments are decorators that accept configuration values.  
They are implemented as a function that returns a decorator, which then wraps the target function.  

This pattern is used when you want to customize decorator behavior, such as setting limits, delays, or retries.


### Example 1: Repeat Decorator with Count

In [1]:
from functools import wraps

def repeat(times):
    """"Decorator factory that repeats function execution"""
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            for _ in range(times):
                result = func(*args, **kwargs)
            return result   # return last result
        return wrapper
    return decorator

@repeat(times=3)
def greet(name):
    print(f"Hello, {name}")

greet("Alice")

Hello, Alice
Hello, Alice
Hello, Alice


### Example 2: Rate Limiter

In [3]:
import time
from functools import wraps

def rate_limit(max_calls, period):
    """
    Limits how many times a function can be called within a given time period.

    max_calls : maximum number of allowed calls
    period    : time window in seconds
    """

    # this list stores timestamps of previous function calls
    calls = []

    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):

            # get the current time
            now = time.time()

            # step 1: remove old calls that are outside the time period
            valid_calls = []
            for call_time in calls:
                if now - call_time < period:
                    valid_calls.append(call_time)

            # update the calls list with only valid timestamps
            calls.clear()
            for call_time in valid_calls:
                calls.append(call_time)

            # step 2: check if the rate limit is exceeded
            if len(calls) >= max_calls:
                raise Exception(
                    "Rate limit exceeded: "
                    + str(max_calls)
                    + " calls per "
                    + str(period)
                    + " seconds"
                )

            # step 3: Record the current call
            calls.append(now)

            # step 4: Call the original function
            return func(*args, **kwargs)

        return wrapper

    return decorator

@rate_limit(max_calls=3, period=5)
def api_call():
    print("API Called")


In [4]:
import time

print("Calling API 3 times quickly:")
api_call()
api_call()
api_call()

print("\nCalling API 4th time (should fail):")
try:
    api_call()
except Exception as e:
    print("Error:", e)

print("\nWaiting 5 seconds...")
time.sleep(5)

print("\nCalling API again (should work):")
api_call()

Calling API 3 times quickly:
API Called
API Called
API Called

Calling API 4th time (should fail):
Error: Rate limit exceeded: 3 calls per 5 seconds

Waiting 5 seconds...

Calling API again (should work):
API Called


#### Dry Run (State Changes Only)

Assume:
- `max_calls = 3`
- `period = 5 seconds`
- `calls = []`

---

#### Call 1 at `t = 100`

```text
calls before = []
calls after  = [100]
→ allowed
```

---

#### Call 2 at `t = 101`

```text
calls before = [100]
valid calls  = [100]
calls after  = [100, 101]
→ allowed
```

---

#### Call 3 at `t = 102`

```text
calls before = [100, 101]
valid calls  = [100, 101]
calls after  = [100, 101, 102]
→ allowed
```

---

#### Call 4 at `t = 103`

```text
calls before = [100, 101, 102]
valid calls  = [100, 101, 102]
len(calls)   = 3 ≥ max_calls
→ exception raised (rate limit exceeded)
```

---

#### Call After Waiting (`t = 108`)

```text
calls before = [100, 101, 102]
valid calls  = []
calls after  = [108]
→ allowed
```

---

#### Key Observations

* `calls` persists because of closure
* Old timestamps are removed before checking
* Function execution happens only if limit allows



### Example 3: Retry Decorator

In [7]:
import time
from functools import wraps

def retry(max_attempts=3, delay=1):
    """Retry failed function calls"""
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            attempts = 0
            while attempts < max_attempts:
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    attempts += 1
                    if attempts >= max_attempts:
                        raise
                    print(f"Attempt {attempts} failed: {e}. Retrying in {delay}s...")
                    time.sleep(delay)
        return wrapper
    return decorator

attempt_counter = {"count": 0}

@retry(max_attempts=3, delay=1)
def unreliable_function():
    attempt_counter["count"] += 1

    if attempt_counter["count"] < 3:
        raise Exception("Forced failure")

    return "Success"



In [8]:
try:
    result = unreliable_function()
    print("Result:", result)
except Exception as e:
    print("Final failure:", e)

Attempt 1 failed: Forced failure. Retrying in 1s...
Attempt 2 failed: Forced failure. Retrying in 1s...
Result: Success


In [10]:
@retry(max_attempts=3, delay=1)
def always_fail():
    raise Exception("Always fails")

always_fail()


Attempt 1 failed: Always fails. Retrying in 1s...
Attempt 2 failed: Always fails. Retrying in 1s...


Exception: Always fails

## Class-Based Decorators

A class-based decorator is a decorator implemented using a class instead of a function.  
The class receives the function during initialization and uses the `__call__` method to add behavior when the function is executed.  

Class-based decorators are useful when the decorator needs to store state or configuration across function calls.


### Count Calls

In [11]:
from functools import wraps

class CountCalls:
    """Decorator that counts function calls"""

    def __init__(self, func):
        wraps(func)(self)
        self.func = func
        self.count = 0

    def __call__(self, *args, **kwargs):
        self.count += 1
        print(f"Call {self.count} of {self.func.__name__}")
        return self.func(*args, **kwargs)
    
@CountCalls
def say_hello():
    print("Hello")

say_hello()
say_hello()

Call 1 of say_hello
Hello
Call 2 of say_hello
Hello


### Class-Based Decorator with Arguments

In [13]:
from functools import wraps

class Retry:
    """Class-based retry decorator with configuration"""

    def __init__(self, max_attempts=3, delay=1):
        self.max_attempts = max_attempts
        self.delay = delay

    def __call__(self, func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            attempts = 0
            while attempts < self.max_attempts:
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    attempts += 1
                    if attempts >= self.max_attempts:
                        raise
                    time.sleep(self.delay)
        return wrapper

attempt_state = {"count": 0}

@Retry(max_attempts=5, delay=1)
def flaky_operation():
    attempt_state["count"] += 1
    print(f"Attempt {attempt_state['count']}")

    if attempt_state["count"] < 3:
        raise Exception("Forced failure")

    return "Success"


In [14]:
try:
    result = flaky_operation()
    print("Result:", result)
except Exception as e:
    print("Final failure:", e)


Attempt 1
Attempt 2
Attempt 3
Result: Success


## Method Decorators

Method decorators are decorators applied to class methods instead of standalone functions.
They work the same way as function decorators, with one important difference:
methods automatically receive self as their first argument.


In [16]:
import time
from functools import wraps

def method_timer(func):
    """Timer decorator for methods"""
    @wraps(func)
    def wrapper(self, *args, **kwargs):
        start = time.time()
        result = func(self, *args, **kwargs)
        end = time.time()
        print(f"{func.__name__} took {end - start:.4f}s")
        return result
    return wrapper

class Calculator:
    @method_timer
    def slow_calculation(self, x):
        time.sleep(1)
        return x*2


In [17]:
calc = Calculator()
result = calc.slow_calculation(5)
print("Result:", result)

slow_calculation took 1.0026s
Result: 10


## Preserving Decorators Information

This example shows how to preserve a function’s original information when using decorators.  
By using `functools.wraps`, the decorated function keeps its name, docstring, and other metadata.  
The `inspect` module can then be used to access the original function’s signature, which is useful for debugging, logging, and introspection in notebooks.


In [18]:
from functools import wraps
import inspect

def smart_decorator(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        # access original function's signature
        sig = inspect.signature(func)
        print(f"Function signature : {sig}")
        return func(*args, **kwargs)
    return wrapper

@smart_decorator
def add(a, b):
    return a + b

result = add(3, 5)
print("Result:", result)

Function signature : (a, b)
Result: 8


## Decorator Chaining Best Practices

This section explains how multiple decorators work together when stacked on a function.  
Decorators are applied from bottom to top, so their order directly affects execution.  
Following a consistent ordering pattern helps keep behavior predictable and readable in notebooks.


In [19]:
from functools import wraps
import time

def validate(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print("validate: checking inputs")
        return func(*args, **kwargs)
    return wrapper

def logger(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print("logger: before function")
        result = func(*args, **kwargs)
        print("logger: after function")
        return result
    return wrapper

def timer(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        end = time.time()
        print(f"timer: took {end - start:.4f}s")
        return result
    return wrapper

@timer        # applied last (outermost)
@logger       # applied second
@validate     # applied first (innermost)
def my_function():
    print("my_function: executing")

my_function()


logger: before function
validate: checking inputs
my_function: executing
logger: after function
timer: took 0.0003s


#### Call flow

```bash
timer.wrapper
  └── logger.wrapper
        └── validate.wrapper
              └── my_function

```

#### Step-by-step execution (exact match to your output)

1.  `timer.wrapper` starts  
    (does NOT print yet)
    
2.  `logger.wrapper` starts  
    → prints
    
    ```
    logger: before function
    
    ```
    
3.  `validate.wrapper` runs  
    → prints
    
    ```
    validate: checking inputs
    
    ```
    
4.  `my_function` runs  
    → prints
    
    ```
    my_function: executing
    
    ```
    
5.  Return back to `logger.wrapper`  
    → prints
    
    ```
    logger: after function
    
    ```
    
6.  Return back to `timer.wrapper`  
    → prints
    
    ```
    timer: took 0.0003s
    
    ```
    



## Common Patterns


### 1. Authentication Decorator

Ensure that a function is executed only if the user is authenticated.

In [22]:
from functools import wraps

# mock auth check
def user_is_authenticated():
    return False  # change to True to allow access

def require_auth(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        if not user_is_authenticated():
            raise PermissionError("Authentication required")
        return func(*args, **kwargs)
    return wrapper

@require_auth
def view_dashboard():
    print("Dashboard loaded")

# run
view_dashboard()


PermissionError: Authentication required

>> Change False → True to see the function execute.

### 2. Permission Decorator (Decorator with Arguments)

Restrict access to a function based on specific permissions (e.g., admin, editor).

In [28]:
from functools import wraps

# mock permission check
def user_has_permission(permission):
    return permission == "admin"  # try changing this. 

def require_permission(permission):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            if not user_has_permission(permission):
                raise PermissionError(f"Permission '{permission}' required")
            return func(*args, **kwargs)
        return wrapper
    return decorator

@require_permission("admin")    
def delete_user(user_id):
    print(f"User {user_id} deleted")

# run
delete_user(42)


User 42 deleted


>> Change "admin" to "user" to see it fail

### 3. Deprecation Warning Decorator

Warn developers that a function should no longer be used and may be removed later.



In [29]:
import warnings
from functools import wraps

# show deprecation warnings in notebook
warnings.simplefilter("always", DeprecationWarning)

def deprecated(replacement=None):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            msg = f"{func.__name__} is deprecated"
            if replacement:
                msg += f", use {replacement} instead"
            warnings.warn(msg, DeprecationWarning, stacklevel=2)
            return func(*args, **kwargs)
        return wrapper
    return decorator

@deprecated(replacement="new_function")
def old_function():
    print("Old function executed")

# run
old_function()

Old function executed


/var/folders/mt/qblgswcj4rs5ll70_77j9_nw0000gn/T/ipykernel_52250/2101039363.py:24: DeprecationWarning: old_function is deprecated, use new_function instead
  old_function()
